# Model Training
Define the target and sensor features for your dataset, split the data without leakage, train a baseline estimator, and persist the selected model.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_PATH = PROJECT_ROOT / 'models' / 'irrigation_model.joblib'

In [ ]:
# Set these to the real target and feature columns after inspecting your dataset.
TARGET_COLUMN = 'replace_with_target'
processed_files = sorted(PROCESSED_DATA_DIR.glob('*.csv'))
if not processed_files:
    raise FileNotFoundError('Run notebook 02 to create a processed CSV first.')
data = pd.read_csv(processed_files[0])
if TARGET_COLUMN not in data.columns:
    raise ValueError(f'Set TARGET_COLUMN to a column in the dataset: {list(data.columns)}')
X = data.drop(columns=[TARGET_COLUMN]).select_dtypes(include='number')
y = data[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = RandomForestRegressor(random_state=42) if y.dtype.kind in 'if' else RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)
print(f'Saved model to {MODEL_PATH}; held-out rows: {len(X_test)}')